# Etap II — Wielowarstwowe Modelowanie Topologiczne (Network Science)

Ten notebook realizuje **Etap II** z `realization_plan.md` dla 3 miast (Dublin, Paryż, Warszawa).

Budujemy porównywalne grafy sieci transportu zbiorowego w dwóch reprezentacjach:

- **L-space**: krawędzie między kolejnymi przystankami w ramach kursu (proxy topologii przebiegu).
- **P-space**: połączenia „bez przesiadki” wynikające ze współobsługi przystanków przez tę samą usługę (domyślnie: **po `route_id`**).

**Wejścia (kanoniczne):** wyłącznie artefakty z Etapu I (`../outputs/etap1/artifacts_index.json`).

**Wyjścia:** `../outputs/etap2/<city>/...` zgodnie z kontraktem z `realization_plan.md`.

Założenia (minimum ryzyk):
- Domyślnie **nie filtrujemy** feedów do jednej daty (Tryb A). Filtrowanie jest opcjonalne (Tryb B).
- Węzeł = `stop_id` (bez agresywnego grupowania do stacji).
- P-space budujemy po `route_id` (stabilne i mniej gęste niż po `trip_id`).


In [1]:
from __future__ import annotations

import json
import os
from collections import Counter
from dataclasses import dataclass
from datetime import datetime
from pathlib import Path
from typing import Dict, List, Optional, Tuple

import numpy as np
import pandas as pd

# GIS do agregacji przestrzennej (most do Etapu IV/V)
import geopandas as gpd

# Metryki grafowe
import networkx as nx

pd.set_option('display.max_columns', 200)
pd.set_option('display.width', 140)

print('Python:', os.popen('python -V').read().strip())
print('pandas:', pd.__version__)
print('numpy:', np.__version__)
print('geopandas:', gpd.__version__)
print('networkx:', nx.__version__)


Python: Python 3.9.13
pandas: 2.3.3
numpy: 2.0.2
geopandas: 1.0.1
networkx: 3.2.1


## 0) Konfiguracja (kontrakt ścieżek) + parametry analizy

Parametry:
- `analysis_date_yyyymmdd` i `analysis_time_window` aktywują **Tryb B**. W przeciwnym razie używamy **Trybu A**.
- `P_SPACE_MODE` steruje definicją P-space (domyślnie `route_id`).

Uwaga: Etap I mógł zapisać ścieżki absolutne w `artifacts_index.json` (np. z innego środowiska). Wprowadzamy mapowanie „fallback” do lokalnego `../outputs/etap1/...`.


In [2]:
ROOT = Path('..').resolve()
OUT_DIR = ROOT / 'outputs' / 'etap2'
OUT_DIR.mkdir(parents=True, exist_ok=True)

ETAP1_DIR = ROOT / 'outputs' / 'etap1'
ARTIFACTS_INDEX_PATH = ETAP1_DIR / 'artifacts_index.json'

if not ARTIFACTS_INDEX_PATH.exists():
    raise FileNotFoundError(f'Missing Etap1 artifacts index: {ARTIFACTS_INDEX_PATH}')

# --- Parametry analizy ---
ANALYSIS_DATE_YYYYMMDD: Optional[str] = None  # np. '20260108' (Tryb B). None => Tryb A.
ANALYSIS_TIME_WINDOW: Optional[Tuple[str, str]] = None  # np. ('07:00:00','10:00:00') obecnie tylko jako metadana/QC

P_SPACE_MODE: str = 'route_id'  # 'route_id' (domyślnie) albo eksperymentalnie: 'trip_id'

# liczba stopów na „linii” powyżej której P-space może eksplodować (QC ostrzeżenie)
P_SPACE_ROUTE_STOP_CAP_WARNING = 500

print('ROOT:', ROOT)
print('ETAP1_DIR:', ETAP1_DIR)
print('OUT_DIR:', OUT_DIR)
print('ANALYSIS_DATE_YYYYMMDD:', ANALYSIS_DATE_YYYYMMDD)
print('ANALYSIS_TIME_WINDOW:', ANALYSIS_TIME_WINDOW)
print('P_SPACE_MODE:', P_SPACE_MODE)


ROOT: C:\Users\Michc\Dropbox\IO_UW\Magisterka\Masters
ETAP1_DIR: C:\Users\Michc\Dropbox\IO_UW\Magisterka\Masters\outputs\etap1
OUT_DIR: C:\Users\Michc\Dropbox\IO_UW\Magisterka\Masters\outputs\etap2
ANALYSIS_DATE_YYYYMMDD: None
ANALYSIS_TIME_WINDOW: None
P_SPACE_MODE: route_id


In [3]:
artifacts_index = json.loads(ARTIFACTS_INDEX_PATH.read_text(encoding='utf-8'))


In [4]:
def resolve_artifact_path(p: str) -> Path:
    """Mapuje ścieżkę z artifacts_index.json na istniejący plik lokalny.

    Zasada:
    - jeżeli ścieżka istnieje -> zwracamy ją
    - wpp. próbujemy znaleźć fragment 'outputs/etap1' i podmienić prefiks na lokalne ROOT
    """
    path = Path(p)
    if path.exists():
        return path

    # próba mapowania na lokalny outputs/etap1
    s = str(p).replace('\\', '/')
    marker = '/outputs/etap1/'
    if marker in s:
        suffix = s.split(marker, 1)[1]
        candidate = ROOT / 'outputs' / 'etap1' / suffix
        if candidate.exists():
            return candidate

    # druga próba: jeżeli w indeksie zapisano względne ścieżki
    candidate = (ROOT / p).resolve()
    if candidate.exists():
        return candidate

    raise FileNotFoundError(f'Cannot resolve artifact path: {p}')


In [5]:
CITIES = ['dublin', 'paris', 'warsaw']

# minimalny kontrakt: tabele GTFS
REQ_GTFS_TABLES = ['stops', 'routes', 'trips', 'stop_times']

# CRS metryczne jak w Etapie I (dla agregacji przestrzennej)
CITY_CRS_METRIC = {
    'dublin': 'EPSG:2157',
    'paris': 'EPSG:2154',
    'warsaw': 'EPSG:2180',
}


In [6]:
@dataclass(frozen=True)
class CityInputs:
    city: str
    gtfs_tables: Dict[str, Path]
    spatial_layers: Dict[str, Path]
    gtfs_report_path: Optional[Path] = None


In [7]:
def build_inputs_from_artifacts_index(artifacts: dict, city: str) -> CityInputs:
    gtfs_tables_raw = artifacts.get('gtfs', {}).get(city, {}).get('tables', {})
    spatial_layers_raw = artifacts.get('spatial', {}).get(city, {}).get('layers', {})

    if not gtfs_tables_raw:
        raise KeyError(f'No GTFS tables listed for city={city} in artifacts_index.json')

    gtfs_tables: Dict[str, Path] = {}
    for t in REQ_GTFS_TABLES:
        if t not in gtfs_tables_raw:
            raise KeyError(f'{city}: missing required GTFS table in artifacts index: {t}')
    for k, p in gtfs_tables_raw.items():
        try:
            gtfs_tables[k] = resolve_artifact_path(p)
        except FileNotFoundError:
            # tablice opcjonalne mogą nie istnieć
            if k in REQ_GTFS_TABLES:
                raise

    spatial_layers: Dict[str, Path] = {}
    for layer_name, layer_paths in spatial_layers_raw.items():
        # zapis z Etapu I ma strukturę: {raw_geojson, metric_geojson, metric_wkt_csv}
        if isinstance(layer_paths, dict) and 'metric_geojson' in layer_paths:
            spatial_layers[layer_name] = resolve_artifact_path(layer_paths['metric_geojson'])

    # [UPDATE] Pobieramy ścieżkę do raportu GTFS, aby wydobyć boundary_path
    gtfs_report_path_raw = artifacts.get('gtfs', {}).get(city, {}).get('report_json')
    gtfs_report_path = resolve_artifact_path(gtfs_report_path_raw) if gtfs_report_path_raw else None

    return CityInputs(city=city, gtfs_tables=gtfs_tables, spatial_layers=spatial_layers, gtfs_report_path=gtfs_report_path)


In [8]:
inputs_by_city: Dict[str, CityInputs] = {city: build_inputs_from_artifacts_index(artifacts_index, city) for city in CITIES}
inputs_by_city


{'dublin': CityInputs(city='dublin', gtfs_tables={'agency': WindowsPath('C:/Users/Michc/Dropbox/IO_UW/Magisterka/Masters/outputs/etap1/dublin/gtfs_normalized/agency.csv'), 'calendar': WindowsPath('C:/Users/Michc/Dropbox/IO_UW/Magisterka/Masters/outputs/etap1/dublin/gtfs_normalized/calendar.csv'), 'calendar_dates': WindowsPath('C:/Users/Michc/Dropbox/IO_UW/Magisterka/Masters/outputs/etap1/dublin/gtfs_normalized/calendar_dates.csv'), 'feed_info': WindowsPath('C:/Users/Michc/Dropbox/IO_UW/Magisterka/Masters/outputs/etap1/dublin/gtfs_normalized/feed_info.csv'), 'routes': WindowsPath('C:/Users/Michc/Dropbox/IO_UW/Magisterka/Masters/outputs/etap1/dublin/gtfs_normalized/routes.csv'), 'shapes': WindowsPath('C:/Users/Michc/Dropbox/IO_UW/Magisterka/Masters/outputs/etap1/dublin/gtfs_normalized/shapes.csv'), 'stop_times': WindowsPath('C:/Users/Michc/Dropbox/IO_UW/Magisterka/Masters/outputs/etap1/dublin/gtfs_normalized/stop_times.csv'), 'stops': WindowsPath('C:/Users/Michc/Dropbox/IO_UW/Magisterka/

In [9]:
def write_run_info_and_inventory(city: str, inputs: CityInputs) -> None:
    city_dir = OUT_DIR / city
    city_dir.mkdir(parents=True, exist_ok=True)

    run_info = {
        'run_utc': datetime.utcnow().isoformat(timespec='seconds') + 'Z',
        'root': str(ROOT),
        'out_dir': str(OUT_DIR),
        'city': city,
        'analysis': {
            'analysis_date_yyyymmdd': ANALYSIS_DATE_YYYYMMDD,
            'analysis_time_window': ANALYSIS_TIME_WINDOW,
            'p_space_mode': P_SPACE_MODE,
        },
        'versions': {
            'python': os.popen('python -V').read().strip(),
            'pandas': pd.__version__,
            'numpy': np.__version__,
            'geopandas': gpd.__version__,
            'networkx': nx.__version__,
        },
    }
    (city_dir / 'run_info.json').write_text(json.dumps(run_info, ensure_ascii=False, indent=2), encoding='utf-8')

    inv_rows = []
    for name, path in sorted(inputs.gtfs_tables.items()):
        inv_rows.append({'city': city, 'type': 'gtfs_table', 'name': name, 'path': str(path), 'exists': path.exists()})
    for name, path in sorted(inputs.spatial_layers.items()):
        inv_rows.append({'city': city, 'type': 'spatial_layer_metric', 'name': name, 'path': str(path), 'exists': path.exists()})

    pd.DataFrame(inv_rows).to_csv(city_dir / 'inventory.csv', index=False)

for city in CITIES:
    write_run_info_and_inventory(city, inputs_by_city[city])


## 1) Wczytanie tabel GTFS (z Etapu I) + sanity-check

Walidujemy minimalnie:
- pokrycie kluczy obcych (`stop_times.stop_id` w `stops`, `stop_times.trip_id` w `trips`, `trips.route_id` w `routes`)
- duplikaty `(trip_id, stop_sequence)`
- udział braków w koordynatach przystanków

Wynik zapisujemy do `outputs/etap2/<city>/qc/validation_report.json`.


In [10]:
def _safe_float(series: pd.Series) -> pd.Series:
    return pd.to_numeric(series.astype('string').str.strip().replace({'': pd.NA}), errors='coerce')

def _safe_int(series: pd.Series) -> pd.Series:
    return pd.to_numeric(series.astype('string').str.strip().replace({'': pd.NA}), errors='coerce').astype('Int64')


In [11]:
def read_gtfs_tables_for_city(inputs: CityInputs) -> Dict[str, pd.DataFrame]:
    tables: Dict[str, pd.DataFrame] = {}

    # kluczowe tabele
    for t in REQ_GTFS_TABLES:
        path = inputs.gtfs_tables[t]
        tables[t] = pd.read_csv(path, dtype='string', keep_default_na=False)

    # opcjonalne
    for opt in ['calendar', 'calendar_dates', 'frequencies', 'transfers', 'pathways']:
        if opt in inputs.gtfs_tables:
            tables[opt] = pd.read_csv(inputs.gtfs_tables[opt], dtype='string', keep_default_na=False)

    # typy minimalne
    stops = tables['stops']
    if 'stop_lat' in stops.columns: stops['stop_lat'] = _safe_float(stops['stop_lat'])
    if 'stop_lon' in stops.columns: stops['stop_lon'] = _safe_float(stops['stop_lon'])
    tables['stops'] = stops

    routes = tables['routes']
    if 'route_type' in routes.columns: routes['route_type'] = _safe_int(routes['route_type'])
    tables['routes'] = routes

    st = tables['stop_times']
    if 'stop_sequence' in st.columns: st['stop_sequence'] = _safe_int(st['stop_sequence'])
    tables['stop_times'] = st

    return tables


In [12]:
def gtfs_qc_report(gtfs: Dict[str, pd.DataFrame]) -> Dict[str, object]:
    report: Dict[str, object] = {}
    report['row_counts'] = {k: int(len(v)) for k, v in gtfs.items()}

    stops = gtfs['stops']
    routes = gtfs['routes']
    trips = gtfs['trips']
    st = gtfs['stop_times']

    cov = {
        'stop_times.stop_id in stops.stop_id': float(st['stop_id'].isin(stops['stop_id']).mean()) if 'stop_id' in st.columns else None,
        'stop_times.trip_id in trips.trip_id': float(st['trip_id'].isin(trips['trip_id']).mean()) if 'trip_id' in st.columns else None,
        'trips.route_id in routes.route_id': float(trips['route_id'].isin(routes['route_id']).mean()) if 'route_id' in trips.columns else None,
    }
    report['coverage'] = cov

    report['stops_coord_nonnull_share'] = {
        'stop_lat': float(stops['stop_lat'].notna().mean()) if 'stop_lat' in stops.columns else None,
        'stop_lon': float(stops['stop_lon'].notna().mean()) if 'stop_lon' in stops.columns else None,
    }

    if {'trip_id', 'stop_sequence'}.issubset(st.columns):
        report['stop_times_duplicate_trip_stop_sequence'] = int(st.duplicated(subset=['trip_id', 'stop_sequence']).sum())
        report['stop_times_stop_sequence_nonnull_share'] = float(st['stop_sequence'].notna().mean())

    return report


In [13]:
city_gtfs: Dict[str, Dict[str, pd.DataFrame]] = {}
city_qc: Dict[str, Dict[str, object]] = {}

for city in CITIES:
    gtfs = read_gtfs_tables_for_city(inputs_by_city[city])
    city_gtfs[city] = gtfs

    qc = gtfs_qc_report(gtfs)
    city_qc[city] = qc

    city_dir = OUT_DIR / city
    qc_dir = city_dir / 'qc'
    qc_dir.mkdir(parents=True, exist_ok=True)
    (qc_dir / 'validation_report.json').write_text(json.dumps(qc, ensure_ascii=False, indent=2), encoding='utf-8')

pd.DataFrame([{ 'city': c, **city_qc[c]['coverage'], **city_qc[c]['row_counts'] } for c in CITIES])


,city,stop_times.stop_id in stops.stop_id,stop_times.trip_id in trips.trip_id,trips.route_id in routes.route_id,stops,routes,trips,stop_times,calendar,calendar_dates,transfers,pathways
0,dublin,1.0,1.0,1.0,5897,238,91869,3452167,139,467,NaN,NaN
1,paris,1.0,1.0,1.0,20635,648,295583,6482907,1032,2424,199874.0,4913.0
2,warsaw,1.0,1.0,1.0,6911,2863,279276,7666946,11040,407089,140.0,NaN


## 2) (Opcjonalnie) wybór horyzontu analizy — Tryb B (filtrowanie do daty)

W Trybie A nic nie filtrujemy.

W Trybie B filtrujemy `trips` (po `service_id`) i następnie `stop_times` do wybranych kursów.
Obsługa Warszawy: feed może nie mieć `calendar.csv` (tylko `calendar_dates.csv`).


In [14]:
def _parse_yyyymmdd(date_yyyymmdd: str) -> datetime:
    return datetime.strptime(date_yyyymmdd, '%Y%m%d')


In [15]:
def select_service_ids_for_date(
    calendar: Optional[pd.DataFrame],
    calendar_dates: Optional[pd.DataFrame],
    date_yyyymmdd: str,
) -> Tuple[set, Dict[str, object]]:
    """Zwraca zestaw service_id aktywnych w danej dacie.

    - calendar: standardowy zakres tygodniowy
    - calendar_dates: wyjątki (add/remove)

    Dla feedów bez calendar (np. Warszawa), opieramy się tylko o calendar_dates.
    """
    date_dt = _parse_yyyymmdd(date_yyyymmdd)
    weekday = ['monday', 'tuesday', 'wednesday', 'thursday', 'friday', 'saturday', 'sunday'][date_dt.weekday()]

    active = set()
    dbg = {'date': date_yyyymmdd, 'weekday': weekday, 'mode': None, 'n_added': 0, 'n_removed': 0}

    if calendar is not None and len(calendar) > 0:
        dbg['mode'] = 'calendar+calendar_dates'
        cal = calendar.copy()
        # daty w GTFS są zwykle w formacie yyyymmdd
        for col in ['start_date', 'end_date']:
            if col in cal.columns:
                cal[col] = cal[col].astype('string')

        if weekday not in cal.columns:
            # brak kolumn dnia tygodnia -> traktujemy calendar jako nieużyteczny
            cal = cal.iloc[0:0]
        else:
            in_range = (cal['start_date'] <= date_yyyymmdd) & (cal['end_date'] >= date_yyyymmdd)
            weekday_on = cal[weekday].astype('string').isin(['1', 'true', 'True', 'YES', 'yes'])
            active = set(cal.loc[in_range & weekday_on, 'service_id'].astype('string')) if 'service_id' in cal.columns else set()

    else:
        dbg['mode'] = 'calendar_dates_only'

    # wyjątki
    if calendar_dates is not None and len(calendar_dates) > 0:
        cd = calendar_dates.copy()
        for col in ['date', 'exception_type']:
            if col in cd.columns:
                cd[col] = cd[col].astype('string')

        if 'date' in cd.columns and 'service_id' in cd.columns:
            today = cd[cd['date'] == date_yyyymmdd]
            if 'exception_type' in today.columns:
                added = set(today.loc[today['exception_type'] == '1', 'service_id'].astype('string'))
                removed = set(today.loc[today['exception_type'] == '2', 'service_id'].astype('string'))
            else:
                # jeśli brak exception_type, traktujemy jako "dodane"
                added = set(today['service_id'].astype('string'))
                removed = set()
            active |= added
            active -= removed
            dbg['n_added'] = len(added)
            dbg['n_removed'] = len(removed)

    return active, dbg


In [16]:
def filter_gtfs_to_date(gtfs: Dict[str, pd.DataFrame], date_yyyymmdd: str) -> Tuple[Dict[str, pd.DataFrame], Dict[str, object]]:
    calendar = gtfs.get('calendar')
    calendar_dates = gtfs.get('calendar_dates')

    active_services, dbg = select_service_ids_for_date(calendar, calendar_dates, date_yyyymmdd)

    trips = gtfs['trips']
    if 'service_id' not in trips.columns:
        # brak service_id -> nie filtrujemy
        return gtfs, {'filtered': False, 'reason': 'trips has no service_id', 'dbg': dbg}

    trips_f = trips[trips['service_id'].isin(list(active_services))].copy()
    trip_ids = set(trips_f['trip_id'].astype('string'))

    st = gtfs['stop_times']
    st_f = st[st['trip_id'].isin(list(trip_ids))].copy()

    out = dict(gtfs)
    out['trips'] = trips_f
    out['stop_times'] = st_f

    meta = {
        'filtered': True,
        'date': date_yyyymmdd,
        'active_services_count': len(active_services),
        'trips_before': int(len(trips)),
        'trips_after': int(len(trips_f)),
        'stop_times_before': int(len(st)),
        'stop_times_after': int(len(st_f)),
        'dbg': dbg,
    }

    return out, meta


In [17]:
filter_meta_by_city: Dict[str, Dict[str, object]] = {}

if ANALYSIS_DATE_YYYYMMDD is not None:
    for city in CITIES:
        gtfs_f, meta = filter_gtfs_to_date(city_gtfs[city], ANALYSIS_DATE_YYYYMMDD)
        city_gtfs[city] = gtfs_f
        filter_meta_by_city[city] = meta

filter_meta_by_city


{}

## 3) Budowa grafów: L-space i P-space

Artefakty:
- `outputs/etap2/<city>/graphs/nodes.csv`
- `outputs/etap2/<city>/graphs/edges_L.csv`
- `outputs/etap2/<city>/graphs/edges_P.csv`

Kontrakt plików:
- `nodes.csv`: `stop_id`, `stop_name`, `stop_lat`, `stop_lon`
- `edges_*.csv`: `u`, `v`, `weight` (dla L-space: liczba wystąpień krawędzi w kursach; dla P-space: liczba wspólnych linii/współwystąpień)


In [18]:
def write_nodes_csv(city: str, stops: pd.DataFrame) -> Path:
    city_dir = OUT_DIR / city
    graphs_dir = city_dir / 'graphs'
    graphs_dir.mkdir(parents=True, exist_ok=True)

    need = ['stop_id', 'stop_name', 'stop_lat', 'stop_lon']
    for c in need:
        if c not in stops.columns:
            stops[c] = pd.NA

    out = stops[need].copy()
    out.to_csv(graphs_dir / 'nodes.csv', index=False)
    return graphs_dir / 'nodes.csv'


In [19]:
def build_edges_l_space(stop_times: pd.DataFrame) -> pd.DataFrame:
    """Edge list L-space: krawędzie między kolejnymi stopami w trip.

    Zwraca DataFrame: u, v, weight.
    """
    st = stop_times.copy()
    if 'stop_sequence' not in st.columns:
        raise ValueError('stop_times missing stop_sequence')

    # porządek
    st = st.sort_values(['trip_id', 'stop_sequence', 'stop_id'], kind='mergesort')

    # nast. stop w trip
    st['u'] = st['stop_id']
    st['v'] = st.groupby('trip_id', sort=False)['stop_id'].shift(-1)

    edges = st.dropna(subset=['v'])[['u', 'v']].copy()
    edges['u'] = edges['u'].astype('string')
    edges['v'] = edges['v'].astype('string')

    # agregacja
    edges = edges.groupby(['u', 'v'], as_index=False).size().rename(columns={'size': 'weight'})
    return edges


In [20]:
def build_edges_p_space_route_id(stop_times: pd.DataFrame, trips: pd.DataFrame) -> Tuple[pd.DataFrame, Dict[str, object]]:
    """P-space po route_id.

    Krawędź pomiędzy dwoma stopami istnieje, jeśli współwystępują na tej samej linii (route_id).
    Waga = liczba linii (route_id), na których współwystępują.

    Implementacja: stop_times -> trips (trip_id->route_id), potem unikalny zbiór stopów per route_id,
    a następnie pary kombinacji.
    """
    st = stop_times[['trip_id', 'stop_id']].copy()
    st['trip_id'] = st['trip_id'].astype('string')
    st['stop_id'] = st['stop_id'].astype('string')

    tt = trips[['trip_id', 'route_id']].copy()
    tt['trip_id'] = tt['trip_id'].astype('string')
    tt['route_id'] = tt['route_id'].astype('string')

    merged = st.merge(tt, on='trip_id', how='left')
    merged = merged.dropna(subset=['route_id'])

    # unikalne stopy na route
    route_to_stops = merged.drop_duplicates(subset=['route_id', 'stop_id']).groupby('route_id')['stop_id'].apply(list)

    # QC: rozmiary
    sizes = route_to_stops.apply(len)
    qc = {
        'routes_with_any_stops': int(len(route_to_stops)),
        'route_stop_count_max': int(sizes.max()) if len(sizes) else 0,
        'routes_over_cap_warning': int((sizes > P_SPACE_ROUTE_STOP_CAP_WARNING).sum()) if len(sizes) else 0,
    }

    # generowanie par
    pair_counter = Counter()
    for stops in route_to_stops:
        if len(stops) < 2:
            continue
        # dla stabilności sort
        stops_sorted = sorted(set(stops))
        # pary nieskierowane
        for i in range(len(stops_sorted)):
            u = stops_sorted[i]
            for j in range(i + 1, len(stops_sorted)):
                v = stops_sorted[j]
                pair_counter[(u, v)] += 1

    edges = pd.DataFrame([(u, v, w) for (u, v), w in pair_counter.items()], columns=['u', 'v', 'weight'])
    return edges, qc


In [21]:
def build_edges_p_space_trip_id(stop_times: pd.DataFrame) -> pd.DataFrame:
    """Eksperymentalnie: P-space po trip_id (bardzo gęste na dużych feedach).

    Krawędź łączy dowolne dwa stop_id, jeśli pojawiają się w tym samym trip.
    Waga = liczba tripów, na których współwystępują.
    """
    st = stop_times[['trip_id', 'stop_id']].copy()
    st['trip_id'] = st['trip_id'].astype('string')
    st['stop_id'] = st['stop_id'].astype('string')

    trip_to_stops = st.drop_duplicates(subset=['trip_id', 'stop_id']).groupby('trip_id')['stop_id'].apply(list)

    pair_counter = Counter()
    for stops in trip_to_stops:
        if len(stops) < 2:
            continue
        stops_sorted = sorted(set(stops))
        for i in range(len(stops_sorted)):
            u = stops_sorted[i]
            for j in range(i + 1, len(stops_sorted)):
                v = stops_sorted[j]
                pair_counter[(u, v)] += 1

    edges = pd.DataFrame([(u, v, w) for (u, v), w in pair_counter.items()], columns=['u', 'v', 'weight'])
    return edges


In [22]:
def write_edges_csv(city: str, name: str, edges: pd.DataFrame) -> Path:
    city_dir = OUT_DIR / city
    graphs_dir = city_dir / 'graphs'
    graphs_dir.mkdir(parents=True, exist_ok=True)

    # kontrakt kolumn
    for c in ['u', 'v', 'weight']:
        if c not in edges.columns:
            raise ValueError(f'edges missing column: {c}')

    edges = edges.copy()
    edges['u'] = edges['u'].astype('string')
    edges['v'] = edges['v'].astype('string')
    edges['weight'] = pd.to_numeric(edges['weight'], errors='coerce').fillna(0).astype('int64')

    out_path = graphs_dir / f'{name}.csv'
    edges.to_csv(out_path, index=False)
    return out_path


In [23]:
graph_qc_by_city: Dict[str, Dict[str, object]] = {}

for city in CITIES:
    gtfs = city_gtfs[city]
    stops = gtfs['stops']
    st = gtfs['stop_times']
    trips = gtfs['trips']

    # nodes
    write_nodes_csv(city, stops)

    # L-space
    edges_l = build_edges_l_space(st)
    write_edges_csv(city, 'edges_L', edges_l)

    # P-space
    p_qc: Dict[str, object] = {}
    if P_SPACE_MODE == 'route_id':
        edges_p, p_qc = build_edges_p_space_route_id(st, trips)
    elif P_SPACE_MODE == 'trip_id':
        edges_p = build_edges_p_space_trip_id(st)
    else:
        raise ValueError(f'Unknown P_SPACE_MODE: {P_SPACE_MODE}')

    write_edges_csv(city, 'edges_P', edges_p)

    graph_qc_by_city[city] = {
        'nodes_written': int(len(stops)),
        'edges_L': int(len(edges_l)),
        'edges_P': int(len(edges_p)),
        **({'p_space_qc': p_qc} if p_qc else {}),
    }

    # [NEW] 3a) Metryki gęstości przestrzennej (Spatial Density)
    # Wykorzystujemy boundary z Etapu I (jeśli dostępne)
    spatial_density = {}
    report_path = inputs_by_city[city].gtfs_report_path
    if report_path and report_path.exists():
        try:
            rep_data = json.loads(report_path.read_text(encoding='utf-8'))
            boundary_path_str = rep_data.get('boundary_path')
            if boundary_path_str:
                boundary_path = resolve_artifact_path(boundary_path_str)
                if boundary_path.exists():
                    gdf_bound = gpd.read_file(boundary_path)
                    # Reprojekcja do metrycznego
                    gdf_bound = gdf_bound.to_crs(CITY_CRS_METRIC[city])
                    area_km2 = gdf_bound.area.sum() / 1e6

                    if area_km2 > 0:
                        spatial_density = {
                            'area_km2': float(area_km2),
                            'stops_per_km2': float(len(stops) / area_km2),
                            'edges_L_per_km2': float(len(edges_l) / area_km2),
                        }
        except Exception as e:
            print(f"[{city}] Spatial density calc failed: {e}")

    graph_qc_by_city[city]['spatial_density'] = spatial_density

graph_qc_by_city


{'dublin': {'nodes_written': 5897,
  'edges_L': 7481,
  'edges_P': 546746,
  'p_space_qc': {'routes_with_any_stops': 238,
   'route_stop_count_max': 195,
   'routes_over_cap_warning': 0},
  'spatial_density': {'area_km2': 6987.721414724326,
   'stops_per_km2': 0.8439088581256274,
   'edges_L_per_km2': 1.0705921939355298}},
 'paris': {'nodes_written': 20635,
  'edges_L': 17911,
  'edges_P': 612344,
  'p_space_qc': {'routes_with_any_stops': 648,
   'route_stop_count_max': 156,
   'routes_over_cap_warning': 0},
  'spatial_density': {'area_km2': 814.8983813889375,
   'stops_per_km2': 25.32217571082799,
   'edges_L_per_km2': 21.979427630561677}},
 'warsaw': {'nodes_written': 6911,
  'edges_L': 9260,
  'edges_P': 536005,
  'p_space_qc': {'routes_with_any_stops': 2863,
   'route_stop_count_max': 198,
   'routes_over_cap_warning': 0},
  'spatial_density': {'area_km2': 5038.321744675431,
   'stops_per_km2': 1.3716869128700726,
   'edges_L_per_km2': 1.8379135889418134}}}

## 4) Metryki grafowe: globalne i per-węzeł

Minimalny zestaw (obowiązkowy):
- global: `n_nodes`, `n_edges`, `n_components`, `largest_component_share`, `density`, `avg_degree`
- per node: `degree` i (opcjonalnie) `strength` (suma wag)

Uwaga: w tej implementacji grafy są traktowane jako **nieskierowane** (co jest typowe dla metryk topologicznych w tym kontekście). L-space powstaje z następstwa w kursie (skierowane), ale dla porównań topologicznych używamy wersji nieskierowanej.


In [24]:
def build_nx_graph_from_edges(edges: pd.DataFrame, weighted: bool = True) -> nx.Graph:
    G = nx.Graph()
    if weighted:
        for u, v, w in edges[['u', 'v', 'weight']].itertuples(index=False, name=None):
            if u == v:
                continue
            # agregacja równoległych krawędzi przebiegała na etapie budowy edge-listy
            G.add_edge(str(u), str(v), weight=float(w))
    else:
        for u, v in edges[['u', 'v']].itertuples(index=False, name=None):
            if u == v:
                continue
            G.add_edge(str(u), str(v))
    return G


In [25]:
def graph_global_metrics(G: nx.Graph) -> Dict[str, float]:
    n = G.number_of_nodes()
    m = G.number_of_edges()

    if n == 0:
        return {
            'n_nodes': 0,
            'n_edges': 0,
            'n_components': 0,
            'largest_component_share': 0.0,
            'density': 0.0,
            'avg_degree': 0.0,
        }

    comps = list(nx.connected_components(G))
    n_comp = len(comps)
    largest = max((len(c) for c in comps), default=0)

    # density = 2m / (n(n-1))
    dens = nx.density(G)
    avg_deg = float(np.mean([d for _, d in G.degree()])) if n > 0 else 0.0

    return {
        'n_nodes': int(n),
        'n_edges': int(m),
        'n_components': int(n_comp),
        'largest_component_share': float(largest / n) if n else 0.0,
        'density': float(dens),
        'avg_degree': float(avg_deg),
    }


In [26]:
def graph_node_metrics(G: nx.Graph) -> pd.DataFrame:
    deg = dict(G.degree())
    strength = dict(G.degree(weight='weight'))
    df = pd.DataFrame({
        'stop_id': pd.Series(list(deg.keys()), dtype='string'),
        'degree': pd.Series(list(deg.values()), dtype='int64'),
        'strength': pd.Series([float(strength[k]) for k in deg.keys()]),
    })
    return df


In [27]:
def write_metrics(city: str, name: str, df: pd.DataFrame) -> Path:
    city_dir = OUT_DIR / city
    metrics_dir = city_dir / 'metrics'
    metrics_dir.mkdir(parents=True, exist_ok=True)
    out_path = metrics_dir / f'{name}.csv'
    df.to_csv(out_path, index=False)
    return out_path


In [28]:
global_rows = []

for city in CITIES:
    city_dir = OUT_DIR / city
    graphs_dir = city_dir / 'graphs'

    edges_l = pd.read_csv(graphs_dir / 'edges_L.csv', dtype={'u': 'string', 'v': 'string', 'weight': 'int64'})
    edges_p = pd.read_csv(graphs_dir / 'edges_P.csv', dtype={'u': 'string', 'v': 'string', 'weight': 'int64'})

    G_l = build_nx_graph_from_edges(edges_l)
    G_p = build_nx_graph_from_edges(edges_p)

    # global
    gl = graph_global_metrics(G_l)
    gp = graph_global_metrics(G_p)

    global_rows.append({'city': city, 'graph': 'L', **gl})
    global_rows.append({'city': city, 'graph': 'P', **gp})

    # node
    n_l = graph_node_metrics(G_l)
    n_p = graph_node_metrics(G_p)
    write_metrics(city, 'stop_metrics_L', n_l)
    write_metrics(city, 'stop_metrics_P', n_p)

    # [NEW] 4a) Diagnostyka różnicowa (Differential Diagnostics L vs P)
    # Porównujemy stopnie w obu przestrzeniach
    # Hipoteza: wysoki stopień w L (hubs fizyczne) vs wysoki w P (hubs przesiadkowe)

    n_l_ren = n_l.rename(columns={'degree': 'degree_L', 'strength': 'strength_L'})
    n_p_ren = n_p.rename(columns={'degree': 'degree_P', 'strength': 'strength_P'})

    diff = pd.merge(n_l_ren, n_p_ren, on='stop_id', how='outer').fillna(0)

    # Wskaźnik "Efficiency Gap": P_degree / L_degree (ile możliwości daje 1 krawędź fizyczna)
    diff['efficiency_ratio'] = diff['degree_P'] / diff['degree_L'].replace(0, 1)

    write_metrics(city, 'differential_diagnostics', diff)

    # Dodajemy kluczowe statystyki do globalnych
    gl['efficiency_ratio_avg'] = float(diff['efficiency_ratio'].mean())
    gl['efficiency_ratio_median'] = float(diff['efficiency_ratio'].median())

    # Update global rows with new metrics (hacky update since gl/gp were already appended)
    # We will just append a specific row for Differential or update the L-space row?
    # Better: Update the last appended row (which was P-space) or L-space?
    # Actually, global_metrics structure is per graph. Let's add 'efficiency_ratio_avg' to the L-space row or a new summary?
    # Let's add it to the 'L' row in global_rows (it is at index -2)
    global_rows[-2]['efficiency_ratio_avg'] = gl['efficiency_ratio_avg'] # Add to L-space context

    # Also add spatial density if available
    sd = graph_qc_by_city.get(city, {}).get('spatial_density', {})
    if sd:
        global_rows[-2]['spatial_density_nodes_km2'] = sd.get('stops_per_km2')

# zapis globalnych
for city in CITIES:
    rows_city = [r for r in global_rows if r['city'] == city]
    write_metrics(city, 'global_metrics', pd.DataFrame(rows_city))

pd.DataFrame(global_rows)


,city,graph,n_nodes,n_edges,n_components,largest_component_share,density,avg_degree,efficiency_ratio_avg,spatial_density_nodes_km2
0,dublin,L,5896,7385,6,0.962178,0.000425,2.505088,74.434791,0.843909
1,dublin,P,5896,546746,5,0.962178,0.031461,185.463365,NaN,NaN
2,paris,L,13981,17650,74,0.881339,0.000181,2.524855,34.510775,25.322176
3,paris,P,14006,612344,49,0.886406,0.006244,87.440240,NaN,NaN
4,warsaw,L,6816,9013,5,0.967576,0.000388,2.644660,59.178697,1.371687
5,warsaw,P,6816,536005,5,0.967576,0.023078,157.278462,NaN,NaN


## 5) Agregacja metryk do jednostek przestrzennych (grid/strefy)

Most do Etapu IV/V: przypisujemy przystanki do poligonów (point-in-polygon) w CRS metrycznym miasta i agregujemy metryki.

- Dublin/Warszawa: `area_metrics.csv` na bazie `grid_1km_metric.geojson`
- Paryż: osobno `area_metrics_grid_1km.csv` (pop grid) oraz `area_metrics_employment_zones.csv` (strefy)

Uwaga: agregacja jest minimalna (count stopów oraz średnia/mediana stopnia). Możesz ją rozbudować w Etapie IV.


In [29]:
def stops_to_gdf_metric(city: str, stops: pd.DataFrame) -> gpd.GeoDataFrame:
    need = ['stop_id', 'stop_lat', 'stop_lon']
    for c in need:
        if c not in stops.columns:
            raise ValueError(f'stops missing {c}')

    gdf = gpd.GeoDataFrame(
        stops[['stop_id', 'stop_lat', 'stop_lon']].copy(),
        geometry=gpd.points_from_xy(stops['stop_lon'], stops['stop_lat']),
        crs='EPSG:4326',
    )
    return gdf.to_crs(CITY_CRS_METRIC[city])


In [30]:
def aggregate_stop_metrics_to_areas(
    city: str,
    areas_path: Path,
    stop_metrics: pd.DataFrame,
    stops: pd.DataFrame,
    area_id_col_hint: Optional[str] = None,
) -> Tuple[pd.DataFrame, Dict[str, object]]:
    areas = gpd.read_file(areas_path)
    if areas.crs is None:
        raise ValueError(f'{city}: areas layer has no CRS: {areas_path}')

    # determinizm CRS
    if str(areas.crs) != CITY_CRS_METRIC[city]:
        areas = areas.to_crs(CITY_CRS_METRIC[city])

    stops_gdf = stops_to_gdf_metric(city, stops)

    # metryki node -> stopy
    sm = stop_metrics.copy()
    sm['stop_id'] = sm['stop_id'].astype('string')

    stops_gdf = stops_gdf.merge(sm, on='stop_id', how='left')

    # identyfikator obszaru
    area_id_col = None
    if area_id_col_hint and area_id_col_hint in areas.columns:
        area_id_col = area_id_col_hint
    else:
        # heurystyka: wybierz pierwszą sensowną kolumnę ID
        candidates = [c for c in ['cell_id', 'id', 'ID', 'GRID_ID', 'grid_id', 'objectid', 'OBJECTID', 'gid'] if c in areas.columns]
        area_id_col = candidates[0] if candidates else None

    if area_id_col is None:
        # awaryjnie: indeks
        areas = areas.reset_index().rename(columns={'index': 'area_id'})
        area_id_col = 'area_id'

    areas_small = areas[[area_id_col, 'geometry']].copy()

    # point-in-polygon
    joined = gpd.sjoin(stops_gdf, areas_small, how='left', predicate='within')

    qc = {
        'areas_n': int(len(areas_small)),
        'stops_n': int(len(stops_gdf)),
        'stops_matched_share': float(joined[area_id_col].notna().mean()),
        'area_id_col': area_id_col,
        'areas_path': str(areas_path),
    }

    # agregacja
    agg = joined.dropna(subset=[area_id_col]).groupby(area_id_col).agg(
        n_stops=('stop_id', 'count'),
        degree_mean=('degree', 'mean'),
        degree_median=('degree', 'median'),
        strength_mean=('strength', 'mean'),
        strength_median=('strength', 'median'),
    ).reset_index().rename(columns={area_id_col: 'area_id'})

    return agg, qc


In [31]:
area_qc_by_city: Dict[str, Dict[str, object]] = {}

for city in CITIES:
    gtfs = city_gtfs[city]
    stops = gtfs['stops']

    metrics_dir = OUT_DIR / city / 'metrics'
    sm_l = pd.read_csv(metrics_dir / 'stop_metrics_L.csv', dtype={'stop_id': 'string'})

    city_layers = inputs_by_city[city].spatial_layers

    # Dublin/Warszawa: grid
    if city in ['dublin', 'warsaw']:
        if 'grid_1km' not in city_layers:
            area_qc_by_city[city] = {'skipped': True, 'reason': 'missing grid_1km layer in artifacts index'}
            continue
        agg, qc = aggregate_stop_metrics_to_areas(city, city_layers['grid_1km'], sm_l, stops)
        write_metrics(city, 'area_metrics', agg)
        area_qc_by_city[city] = qc

    # Paris: grid 1km (pop) + employment zones
    if city == 'paris':
        qc_block = {}
        if 'pop_grid_1km' in city_layers:
            agg1, qc1 = aggregate_stop_metrics_to_areas(city, city_layers['pop_grid_1km'], sm_l, stops, area_id_col_hint='cell_id')
            metrics_dir.mkdir(parents=True, exist_ok=True)
            agg1.to_csv(metrics_dir / 'area_metrics_grid_1km.csv', index=False)
            qc_block['grid_1km'] = qc1
        else:
            qc_block['grid_1km'] = {'skipped': True, 'reason': 'missing pop_grid_1km layer'}

        if 'employment_zones' in city_layers:
            agg2, qc2 = aggregate_stop_metrics_to_areas(city, city_layers['employment_zones'], sm_l, stops)
            agg2.to_csv(metrics_dir / 'area_metrics_employment_zones.csv', index=False)
            qc_block['employment_zones'] = qc2
        else:
            qc_block['employment_zones'] = {'skipped': True, 'reason': 'missing employment_zones layer'}

        area_qc_by_city[city] = qc_block

area_qc_by_city


{'dublin': {'areas_n': 72869,
  'stops_n': 5897,
  'stops_matched_share': 0.9981346447346108,
  'area_id_col': 'OBJECTID',
  'areas_path': 'C:\\Users\\Michc\\Dropbox\\IO_UW\\Magisterka\\Masters\\outputs\\etap1\\dublin\\spatial\\grid_1km_metric.geojson'},
 'paris': {'grid_1km': {'areas_n': 932764,
   'stops_n': 20635,
   'stops_matched_share': 1.0,
   'area_id_col': 'cell_id',
   'areas_path': 'C:\\Users\\Michc\\Dropbox\\IO_UW\\Magisterka\\Masters\\outputs\\etap1\\paris\\spatial\\pop_grid_1km_metric.geojson'},
  'employment_zones': {'areas_n': 39,
   'stops_n': 20635,
   'stops_matched_share': 0.21177610855342863,
   'area_id_col': 'OBJECTID',
   'areas_path': 'C:\\Users\\Michc\\Dropbox\\IO_UW\\Magisterka\\Masters\\outputs\\etap1\\paris\\spatial\\employment_zones_metric.geojson'}},
 'warsaw': {'areas_n': 315857,
  'stops_n': 6911,
  'stops_matched_share': 1.0,
  'area_id_col': 'area_id',
  'areas_path': 'C:\\Users\\Michc\\Dropbox\\IO_UW\\Magisterka\\Masters\\outputs\\etap1\\warsaw\\spat

## 6) QC (ostrzeżenia) + raport `etap2_summary.md`

Generujemy:
- `outputs/etap2/<city>/qc/warnings.md`
- `outputs/etap2/<city>/reports/etap2_summary.md`

Raport jest krótki i replikowalny: parametry + najważniejsze metryki + ostrzeżenia.


In [32]:
def compute_warnings(city: str) -> List[str]:
    warnings: List[str] = []

    qc = city_qc[city]
    cov = qc.get('coverage', {})
    for k, v in cov.items():
        if v is not None and v < 0.999:
            warnings.append(f"Low coverage: {k} = {v:.6f} (expected ~1.0)")

    # graph metrics
    gm = pd.read_csv(OUT_DIR / city / 'metrics' / 'global_metrics.csv')
    for _, row in gm.iterrows():
        g = row['graph']
        if float(row.get('largest_component_share', 0.0)) < 0.5:
            warnings.append(f"{g}-space: largest_component_share low ({row['largest_component_share']:.3f})")
        if float(row.get('density', 0.0)) > 0.05 and g == 'P':
            warnings.append(f"P-space: density high ({row['density']:.6f}) — graph might be too dense; consider route_id mode & caps")

    # P-space route cap warning
    p_qc = graph_qc_by_city.get(city, {}).get('p_space_qc')
    if isinstance(p_qc, dict) and p_qc.get('routes_over_cap_warning', 0) > 0:
        warnings.append(
            f"P-space: {p_qc['routes_over_cap_warning']} routes have > {P_SPACE_ROUTE_STOP_CAP_WARNING} unique stops; P-space may inflate"
        )

    # spatial join coverage
    aqc = area_qc_by_city.get(city)
    if isinstance(aqc, dict) and 'stops_matched_share' in aqc:
        if aqc['stops_matched_share'] < 0.95:
            warnings.append(f"Spatial join: low matched share ({aqc['stops_matched_share']:.3f}) — check CRS/extent")
    if isinstance(aqc, dict) and city == 'paris':
        for name, block in aqc.items():
            if isinstance(block, dict) and 'stops_matched_share' in block and block['stops_matched_share'] < 0.95:
                warnings.append(f"Paris spatial join ({name}): low matched share ({block['stops_matched_share']:.3f})")

    return warnings


In [33]:
# Pomocniczo: bezpieczna tabela Markdown bez zależności od `tabulate`

def _df_to_md_table(df: pd.DataFrame, max_rows: int = 200) -> str:
    """Renderuje DataFrame do Markdown.

    Pandas `to_markdown()` wymaga opcjonalnego pakietu `tabulate`.
    Żeby notebook działał w minimalnym środowisku, robimy fallback.
    """
    df = df.head(max_rows)
    try:
        return df.to_markdown(index=False)
    except Exception:
        cols = list(df.columns)
        header = '| ' + ' | '.join(str(c) for c in cols) + ' |'
        sep = '| ' + ' | '.join(['---'] * len(cols)) + ' |'
        rows = []
        for row in df.itertuples(index=False, name=None):
            rows.append('| ' + ' | '.join('' if (v is None or (isinstance(v, float) and np.isnan(v))) else str(v) for v in row) + ' |')
        return '\n'.join([header, sep] + rows)


In [34]:
def write_city_reports(city: str) -> None:
    city_dir = OUT_DIR / city
    qc_dir = city_dir / 'qc'
    rep_dir = city_dir / 'reports'
    qc_dir.mkdir(parents=True, exist_ok=True)
    rep_dir.mkdir(parents=True, exist_ok=True)

    warnings = compute_warnings(city)
    (qc_dir / 'warnings.md').write_text('\n'.join(f'- {w}' for w in (warnings or ['(no warnings)'])) + '\n', encoding='utf-8')

    gm = pd.read_csv(city_dir / 'metrics' / 'global_metrics.csv')

    lines = []
    lines.append(f"# Etap II — podsumowanie ({city})")
    lines.append('')
    lines.append(f"Uruchomiono (UTC): `{datetime.utcnow().isoformat(timespec='seconds')}Z`")
    lines.append('')
    lines.append('## Parametry')
    lines.append('')
    lines.append(f"- analysis_date_yyyymmdd: `{ANALYSIS_DATE_YYYYMMDD}`")
    lines.append(f"- analysis_time_window: `{ANALYSIS_TIME_WINDOW}`")
    lines.append(f"- p_space_mode: `{P_SPACE_MODE}`")
    lines.append('')
    lines.append('## Metryki globalne')
    lines.append('')
    lines.append(_df_to_md_table(gm))
    lines.append('')
    lines.append('## Wejścia (kanoniczne)')
    lines.append('')
    inv = pd.read_csv(city_dir / 'inventory.csv')
    lines.append(_df_to_md_table(inv))
    lines.append('')
    lines.append('## Ostrzeżenia QC')
    lines.append('')
    if warnings:
        lines.extend([f"- {w}" for w in warnings])
    else:
        lines.append('- (no warnings)')
    lines.append('')

    (rep_dir / 'etap2_summary.md').write_text('\n'.join(lines) + '\n', encoding='utf-8')

for city in CITIES:
    write_city_reports(city)

# szybki podgląd zbiorczy
pd.concat([pd.read_csv(OUT_DIR / c / 'metrics' / 'global_metrics.csv').assign(city=c) for c in CITIES], ignore_index=True)


,city,graph,n_nodes,n_edges,n_components,largest_component_share,density,avg_degree,efficiency_ratio_avg,spatial_density_nodes_km2
0,dublin,L,5896,7385,6,0.962178,0.000425,2.505088,74.434791,0.843909
1,dublin,P,5896,546746,5,0.962178,0.031461,185.463365,NaN,NaN
2,paris,L,13981,17650,74,0.881339,0.000181,2.524855,34.510775,25.322176
3,paris,P,14006,612344,49,0.886406,0.006244,87.440240,NaN,NaN
4,warsaw,L,6816,9013,5,0.967576,0.000388,2.644660,59.178697,1.371687
5,warsaw,P,6816,536005,5,0.967576,0.023078,157.278462,NaN,NaN


## 7) Indeks artefaktów + zbiorczy inwentarz + `summary.md` (zamknięcie Etapu II)

Tak jak w Etapie I, na końcu zapisujemy zbiorcze pliki w `../outputs/etap2/`:
- `run_info.json` (globalny)
- `inventory.csv` (zgrupowany, dla wszystkich miast)
- `artifacts_index.json` (kontrakt wyjść Etapu II)
- `summary.md` (krótka interpretacja liczbowo-jakościowa)


In [35]:
# globalny run_info (repozytoryjny)
run_info_etap2 = {
    'run_utc': datetime.utcnow().isoformat(timespec='seconds') + 'Z',
    'root': str(ROOT),
    'out_dir': str(OUT_DIR),
    'cities': CITIES,
    'analysis': {
        'analysis_date_yyyymmdd': ANALYSIS_DATE_YYYYMMDD,
        'analysis_time_window': ANALYSIS_TIME_WINDOW,
        'p_space_mode': P_SPACE_MODE,
    },
    'versions': {
        'python': os.popen('python -V').read().strip(),
        'pandas': pd.__version__,
        'numpy': np.__version__,
        'geopandas': gpd.__version__,
        'networkx': nx.__version__,
    },
}
(OUT_DIR / 'run_info.json').write_text(json.dumps(run_info_etap2, ensure_ascii=False, indent=2), encoding='utf-8')


520

In [36]:
# zagregowany inventory.csv
inv_all = []
for city in CITIES:
    inv_city_path = OUT_DIR / city / 'inventory.csv'
    if inv_city_path.exists():
        inv_city = pd.read_csv(inv_city_path)
        inv_city['city'] = city
        inv_all.append(inv_city)

if inv_all:
    inv_all_df = pd.concat(inv_all, ignore_index=True)
else:
    inv_all_df = pd.DataFrame(columns=['city', 'type', 'name', 'path', 'exists'])

inv_all_df.to_csv(OUT_DIR / 'inventory.csv', index=False)


In [37]:
# artifacts_index.json (Etap II)

def _maybe_path(p: Path) -> Optional[str]:
    return str(p) if p.exists() else None

artifacts_index_etap2 = {
    'generated_at_utc': run_info_etap2['run_utc'],
    'analysis': run_info_etap2['analysis'],
    'graphs': {},
    'metrics': {},
    'qc': {},
    'reports': {},
}

for city in CITIES:
    city_dir = OUT_DIR / city
    artifacts_index_etap2['graphs'][city] = {
        'nodes_csv': _maybe_path(city_dir / 'graphs' / 'nodes.csv'),
        'edges_L_csv': _maybe_path(city_dir / 'graphs' / 'edges_L.csv'),
        'edges_P_csv': _maybe_path(city_dir / 'graphs' / 'edges_P.csv'),
    }
    artifacts_index_etap2['metrics'][city] = {
        'stop_metrics_L_csv': _maybe_path(city_dir / 'metrics' / 'stop_metrics_L.csv'),
        'stop_metrics_P_csv': _maybe_path(city_dir / 'metrics' / 'stop_metrics_P.csv'),
        'global_metrics_csv': _maybe_path(city_dir / 'metrics' / 'global_metrics.csv'),
        # agregacje przestrzenne
        'area_metrics_csv': _maybe_path(city_dir / 'metrics' / 'area_metrics.csv'),
        'area_metrics_grid_1km_csv': _maybe_path(city_dir / 'metrics' / 'area_metrics_grid_1km.csv'),
        'area_metrics_employment_zones_csv': _maybe_path(city_dir / 'metrics' / 'area_metrics_employment_zones.csv'),
    }
    artifacts_index_etap2['qc'][city] = {
        'validation_report_json': _maybe_path(city_dir / 'qc' / 'validation_report.json'),
        'warnings_md': _maybe_path(city_dir / 'qc' / 'warnings.md'),
    }
    artifacts_index_etap2['reports'][city] = {
        'etap2_summary_md': _maybe_path(city_dir / 'reports' / 'etap2_summary.md'),
    }

(OUT_DIR / 'artifacts_index.json').write_text(json.dumps(artifacts_index_etap2, ensure_ascii=False, indent=2), encoding='utf-8')


4808

In [38]:
# summary.md (zbiorczy)
summary_lines: List[str] = []
summary_lines.append('# Etap II — podsumowanie')
summary_lines.append('')
summary_lines.append(f"Uruchomiono (UTC): `{run_info_etap2['run_utc']}`")
summary_lines.append('')
summary_lines.append('## Parametry')
summary_lines.append('')
summary_lines.append(f"- analysis_date_yyyymmdd: `{ANALYSIS_DATE_YYYYMMDD}`")
summary_lines.append(f"- analysis_time_window: `{ANALYSIS_TIME_WINDOW}`")
summary_lines.append(f"- p_space_mode: `{P_SPACE_MODE}`")
summary_lines.append('')
summary_lines.append('## Metryki globalne (L/P)')
summary_lines.append('')

# global metrics per miasto
for city in CITIES:
    gm_path = OUT_DIR / city / 'metrics' / 'global_metrics.csv'
    summary_lines.append(f"### {city}")
    if gm_path.exists():
        gm = pd.read_csv(gm_path)
        summary_lines.append(_df_to_md_table(gm))
    else:
        summary_lines.append('- (missing global_metrics.csv)')
    summary_lines.append('')

summary_lines.append('## QC — ostrzeżenia (pierwsze linie)')
summary_lines.append('')
for city in CITIES:
    warn_path = OUT_DIR / city / 'qc' / 'warnings.md'
    summary_lines.append(f"### {city}")
    if warn_path.exists():
        # wstawiamy 10 pierwszych linii, żeby summary był krótki
        w_lines = warn_path.read_text(encoding='utf-8').splitlines()[:10]
        summary_lines.extend(w_lines if w_lines else ['- (empty warnings.md)'])
    else:
        summary_lines.append('- (missing warnings.md)')
    summary_lines.append('')

(OUT_DIR / 'summary.md').write_text('\n'.join(summary_lines) + '\n', encoding='utf-8')

# podgląd sanity
{
    'run_info': str(OUT_DIR / 'run_info.json'),
    'inventory': str(OUT_DIR / 'inventory.csv'),
    'artifacts_index': str(OUT_DIR / 'artifacts_index.json'),
    'summary': str(OUT_DIR / 'summary.md'),
}


{'run_info': 'C:\\Users\\Michc\\Dropbox\\IO_UW\\Magisterka\\Masters\\outputs\\etap2\\run_info.json',
 'inventory': 'C:\\Users\\Michc\\Dropbox\\IO_UW\\Magisterka\\Masters\\outputs\\etap2\\inventory.csv',
 'artifacts_index': 'C:\\Users\\Michc\\Dropbox\\IO_UW\\Magisterka\\Masters\\outputs\\etap2\\artifacts_index.json',
 'summary': 'C:\\Users\\Michc\\Dropbox\\IO_UW\\Magisterka\\Masters\\outputs\\etap2\\summary.md'}